# 03 · El Proceso Gaussiano (GP) sobre 20 variables

El GP es la versión más "pura" de incertidumbre sensible a la distancia: su varianza es chica cerca de los datos con los que aprendió y crece al alejarse. El problema es que un GP exacto con 184 mil filas no entra en memoria, así que lo aproximamos con Random Fourier Features (`redes.py`) y, para que las distancias tengan sentido, lo corremos sobre las 20 variables que elige un Lasso.

**Cambio de roles desde acá:** el 10% que en el 01 llamamos `calib` pasa a ser **validación** (para elegir hiperparámetros y para el early stopping), y el 30% `test` pasa a ser el **pool**, donde corre el experimento conformal del 04.

- **Lee:** `outputs/datos_procesados.npz`, `outputs/ids_*.csv`, `outputs/preds/xgboost_*.csv`.
- **Guarda:** `outputs/preds/gp_vars_calib.csv` y `outputs/preds/gp_vars_test.csv` (logit y varianza de cada solicitante).
- **En el informe:** es el "GP" de la Tabla 1, la Tabla 2 y la Figura 2, y el cociente de varianzas de 0,79 de la sección 4.1.

In [ ]:
# Bloque 0. Diagnóstico rápido de lo que guardó el 01: versiones, formas, tipos, si hay NaN y si las
# columnas quedaron centradas. Es solo un chequeo antes de arrancar.
import os
import sys
import numpy as np
import pandas as pd
import torch
print("Python", sys.version.split()[0], "| torch", torch.__version__)   # split()[0]: solo el número de versión
datos = np.load("outputs/datos_procesados.npz", allow_pickle=True)   # allow_pickle por si hay arrays de texto
for clave in datos.files:   # cada matriz guardada (X_train, y_train, ...)
    a = datos[clave]
    print(f"\n{clave}: forma {a.shape}, tipo {a.dtype}")
    if a.ndim == 2 and a.dtype != object:   # solo las matrices numéricas (filas x columnas)
        a = a.astype(np.float32)
        # a.mean(0) = media de cada columna; la mediana de esas medias resume si están centradas en 0
        print("  NaN:", int(np.isnan(a).sum()),
              "| mediana de las medias:", round(float(np.nanmedian(a.mean(0))), 3),
              "| mediana de los desvíos:", round(float(np.nanmedian(a.std(0))), 3))
for f in sorted(os.listdir("outputs")):
    if f.startswith("ids"):   # los CSV con el ID de cada fila de cada partición
        print(f, "->", pd.read_csv(f"outputs/{f}").head(2).to_dict("list"))

# Todo coincide: sin NaN y columnas centradas (mediana de las medias ≈ 0,006; mediana de los desvíos ≈ 0,41, menor a
# 1 porque las 104 dummies de texto quedan en 0/1 y no se escalan).

Python 3.14.2 | torch 2.14.1+cpu

X_train: forma (184506, 200), tipo float32
  NaN: 0 | mediana de las medias: 0.006 | mediana de los desvíos: 0.413

X_calib: forma (30751, 200), tipo float32
  NaN: 0 | mediana de las medias: 0.006 | mediana de los desvíos: 0.414

X_test: forma (92254, 200), tipo float32
  NaN: 0 | mediana de las medias: 0.006 | mediana de los desvíos: 0.412

y_train: forma (184506,), tipo int64

y_calib: forma (30751,), tipo int64

y_test: forma (92254,), tipo int64
ids_calib.csv -> {'SK_ID_CURR': [171439, 384227]}
ids_test.csv -> {'SK_ID_CURR': [253614, 116431]}
ids_train.csv -> {'SK_ID_CURR': [420938, 286525]}


In [5]:
# Bloque 1. Cargamos las matrices del 01 y las predicciones de XGBoost, y comprobamos que la fila i de cada matriz
# sea el mismo solicitante que la fila i del CSV de XGBoost. Esto es clave: XGBoost y el GP se comparan SOBRE LOS
# MISMOS solicitantes, y si una fila estuviera corrida, los p-valores, las tablas y las figuras saldrían mal sin
# avisar. Los assert frenan el código si algo no coincide.
# Recordemos el cambio de roles: train (60%) es donde aprende el GP; el 10% "calib" del 01 ahora es VALIDACIÓN; y
# el 30% "test" es el POOL donde corre el experimento conformal (que lo parte 20 veces en calibración y prueba).

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")   # avisos de sklearn que no cambian nada

import numpy as np
import pandas as pd
import torch
SEED = 42
datos = np.load("outputs/datos_procesados.npz", allow_pickle=True)
X_tr = datos["X_train"].astype(np.float32)   # torch trabaja con float32
y_tr = datos["y_train"].astype(np.float32)
X_va = datos["X_calib"].astype(np.float32)   # el 10%: ahora es VALIDACIÓN
y_va = datos["y_calib"].astype(np.float32)
X_po = datos["X_test"].astype(np.float32)   # el 30%: el POOL (después se parte en calibración + prueba)
y_po = datos["y_test"].astype(np.float32)
ids_va = pd.read_csv("outputs/ids_calib.csv")["SK_ID_CURR"].to_numpy()

ids_po = pd.read_csv("outputs/ids_test.csv")["SK_ID_CURR"].to_numpy()
# las predicciones de XGBoost, reordenadas igual que las filas de X (.loc[ids] las pone en ese orden)
xgb_va = pd.read_csv("outputs/preds/xgboost_calib.csv").set_index("SK_ID_CURR").loc[ids_va]
xgb_po = pd.read_csv("outputs/preds/xgboost_test.csv").set_index("SK_ID_CURR").loc[ids_po]
# chequeo de alineación: si falla, las filas de X y las de los CSV no están en el mismo orden
assert (xgb_va["TARGET"].to_numpy() == y_va).all(), "validación desalineada"
assert (xgb_po["TARGET"].to_numpy() == y_po).all(), "pool desalineado"
thin_va = xgb_va["thin_file"].to_numpy().astype(bool)
thin_po = xgb_po["thin_file"].to_numpy().astype(bool)
# nombres de las columnas, para leer qué variables elige el Lasso
try:
    import joblib
    nombres = np.array(joblib.load("outputs/preproc.joblib").get_feature_names_out())
except Exception as e:
    print("No pude leer los nombres:", e)
    nombres = np.array([f"x{j}" for j in range(X_tr.shape[1])])
print("formas:", X_tr.shape, X_va.shape, X_po.shape, "| nombres:", len(nombres))
print("tasa de mora train/val/pool:",
      y_tr.mean().round(4), y_va.mean().round(4), y_po.mean().round(4))
print("thin-file en el pool:", int(thin_po.sum()),
      "| morosos thin-file en el pool:", int(y_po[thin_po].sum()))
p_xgb = 1 / (1 + np.exp(-xgb_po["logit"].to_numpy()))   # de logit a probabilidad
print("prob. media XGBoost en el pool:", round(float(p_xgb.mean()), 4))

# Mora de 8,07% en train, validación y pool (la partición estratificada funcionó); 13.206 thin-file en el pool, de
# los cuales 1.337 son morosos (10,1%). La probabilidad media de XGBoost (8,11%) se parece a la mora real: está
# calibrado en promedio.

formas: (184506, 200) (30751, 200) (92254, 200) | nombres: 200
tasa de mora train/val/pool: 0.0807 0.0807 0.0807
thin-file en el pool: 13206 | morosos thin-file en el pool: 1337
prob. media XGBoost en el pool: 0.0811


In [ ]:
# Bloque 2. ¿Por qué solo 20 variables? El GP mide qué tan parecidos son dos solicitantes con la distancia entre
# ellos. Con muchas variables, todas las distancias entre pares se parecen ("maldición de la dimensionalidad"): dos
# personas distintas quedan casi a la misma distancia que dos parecidas, y el GP deja de distinguir a nadie. Por eso
# elegimos pocas variables, las más importantes, con una regresión logística con Lasso (L1) sobre una submuestra de train.

from sklearn.linear_model import LogisticRegression

rng=np.random.default_rng(SEED)
idx_sub=rng.choice(len(X_tr), size=min(60000, len(X_tr)), replace=False)   # 60 mil filas de train al azar, sin repetir

# penalty="l1": Lasso (lleva coeficientes exactamente a 0); C = 1/penalización, así que C chico = penaliza más;
# solver="liblinear": el algoritmo que admite L1; max_iter: tope de iteraciones para converger.
lasso=LogisticRegression(penalty="l1", C=0.05, solver="liblinear", max_iter=1000)

lasso.fit(X_tr[idx_sub], y_tr[idx_sub])
coef=lasso.coef_.ravel()   # vector con un coeficiente por variable

# ordenamos las variables de más a menos importante según el valor absoluto de su coeficiente
orden=np.argsort(-np.abs(coef))

print("variables con coeficiente distinto de cero:", int((coef !=0).sum()))   # int() lo pasa a un entero común para que se imprima limpio

for j in orden[:20]: print(f"{nombres[j]:50s}{coef[j]:.3f}")   # las 20 más importantes: nombre y coeficiente

# El FutureWarning de penalty es un aviso de que la opción va a cambiar de nombre en versiones futuras; no cambia nada.
# 82 de las 200 variables quedan con coeficiente distinto de cero. Arriba de todo: AMT_GOODS_PRICE (-0.58) y
# AMT_CREDIT (+0.51), que son casi colineales (el modelo usa su diferencia, así que sus signos no se interpretan por
# separado), y después los puntajes externos EXT_SOURCE_3 y EXT_SOURCE_2, género y educación.
# Ninguna variable del bureau en el top 20, coherente con que los thin-file no tienen bureau.

c:\Users\maxim\OneDrive\Desktop\beyond-credit-score\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


variables con coeficiente distinto de cero: 82
num__AMT_GOODS_PRICE                              -0.583
num__AMT_CREDIT                                   0.506
num__EXT_SOURCE_3                                 -0.456
num__EXT_SOURCE_2                                 -0.377
cat__CODE_GENDER_F                                -0.321
cat__NAME_EDUCATION_TYPE_Higher education         -0.317
num__EXT_SOURCE_1                                 -0.188
cat__NAME_CONTRACT_TYPE_Revolving loans           -0.160
cat__NAME_EDUCATION_TYPE_Incomplete higher        -0.153
cat__ORGANIZATION_TYPE_Self-employed              0.150
num__DAYS_EMPLOYED                                0.148
cat__ORGANIZATION_TYPE_Business Entity Type 3     0.121
cat__NAME_FAMILY_STATUS_Married                   -0.121
cat__ORGANIZATION_TYPE_Construction               0.121
num__missingindicator_OWN_CAR_AGE                 0.120
cat__OCCUPATION_TYPE_Core staff                   -0.115
num__FLAG_DOCUMENT_3                           

In [19]:
# Bloque 3. Adaptador entre numpy y PyTorch: arma los DataLoaders que necesita el entrenamiento, para cualquier
# subconjunto de columnas (las k elegidas por Lasso) y de filas (todo train o la submuestra de 60 mil).
# Un DataLoader corta los datos en batches: grupitos de filas que se procesan juntos (512 en train; 4096 en
# validación, porque ahí no se entrena y conviene ir rápido). Train se mezcla en cada época (shuffle) para que el
# orden no sesgue el aprendizaje, con semilla fija para que sea reproducible; validación no se mezcla porque solo se
# evalúa. TensorDataset empaqueta los pares (variables, etiqueta) y el DataLoader los reparte.

from torch.utils.data import DataLoader, TensorDataset
def armar_loaders(cols, idx_train=None, batch=512):
    """Arma los loaders de train (mezclado) y de validación (en orden), usando solo las columnas cols.
    cols: qué columnas (variables) usar.
    idx_train: qué filas de train usar (None = todas).
    batch: cuántos solicitantes por tanda en train."""
    Xt = X_tr if idx_train is None else X_tr[idx_train]   # todas las filas o solo las pedidas
    yt = y_tr if idx_train is None else y_tr[idx_train]
    # X[:, cols]: todas las filas, solo las columnas cols; torch.from_numpy pasa de numpy a tensor
    ds_tr = TensorDataset(torch.from_numpy(Xt[:, cols]), torch.from_numpy(yt))
    ds_va = TensorDataset(torch.from_numpy(X_va[:, cols]), torch.from_numpy(y_va))
    gen = torch.Generator().manual_seed(SEED)   # mezcla reproducible
    tr = DataLoader(ds_tr, batch_size=batch, shuffle=True, generator=gen)   # train: mezclado en cada época
    va = DataLoader(ds_va, batch_size=4096)   # validación: en orden, tandas grandes
    return tr, va
# prueba de humo antes de gastar cómputo: 5 variables, submuestra de 60k, sacamos la primera tanda y la miramos
tr, va = armar_loaders(orden[:5], idx_train=idx_sub)
x, y = next(iter(tr))   # la primera tanda del loader
print("un batch:", x.shape, x.dtype, "| etiquetas:", y.shape, y.dtype,
      "| morosos en el batch:", int(y.sum()))

# Una tanda de [512, 5] en float32 con 54 morosos (~10%, parecido a la mora general): todo en orden.

un batch: torch.Size([512, 5]) torch.float32 | etiquetas: torch.Size([512]) torch.float32 | morosos en el batch: 54


In [20]:
# Bloque 4. Búsqueda de hiperparámetros del GP en validación: probamos 12 combinaciones (k en {3, 5, 10, 20}
# variables de Lasso x scale en {√k/2, √k, 2√k}), entrenamos cada una con la submuestra de 60 mil y nos quedamos con
# la de menor pérdida de validación. Fijamos la semilla antes de cada modelo para que la comparación sea justa.
# NUNCA se usa el pool: la garantía de Conformal Selection depende de que el score se fije sin mirar esos datos.
#   k     = cuántas variables de Lasso ve el GP. Pocas: distingue bien pero ve poca información; muchas: las
#           distancias se aplanan.
#   scale = la "regla" con la que se miden las distancias (el largo de escala del kernel). Chica: todo parece lejos;
#           grande: todo parece cerca. Por eso la probamos alrededor de √k (con k variables estandarizadas, la
#           distancia típica entre dos personas crece como √k).

import math
import time
from redes import GPVariables, entrenar
LR = 1e-2   # learning rate del GP (es una logística sobre features fijas, así que aguanta un paso grande)
resultados = []   # una fila por combinación (k, factor)
for k in [3, 5, 10, 20]:
    cols = orden[:k]   # las k variables más importantes según Lasso
    tr, va = armar_loaders(cols, idx_train=idx_sub)
    for factor in [0.5, 1, 2]:
        torch.manual_seed(SEED)   # mismas random features iniciales en todas: comparación justa
        modelo = GPVariables(k, scale=factor * math.sqrt(k))   # GP con k entradas y scale = factor x raíz(k)
        t0 = time.time()
        hist = entrenar(modelo, tr, va, lr=LR, verbose=False)   # devuelve una lista de (época, pérdida train, pérdida val)
        # h[2] = pérdida de validación de cada época; guardamos la mínima (la del modelo que se queda)
        resultados.append({"k": k, "factor": factor,
                           "perdida_val": round(min(h[2] for h in hist), 5),
                           "epocas": len(hist), "segundos": round(time.time() - t0)})
        print(resultados[-1])   # [-1] = la última fila agregada
tabla_busqueda = pd.DataFrame(resultados).sort_values("perdida_val")   # de menor a mayor pérdida
print(tabla_busqueda.head())

# Gana k = 20 con scale = 1 x √k (pérdida de validación 0.2495). k pesa mucho más que la escala: con más variables
# el GP ve más información. Ojo: k = 20 es el borde de la grilla; probamos hasta ahí para no caer en el problema de
# la dimensionalidad, y la SNGP (05) es justamente la respuesta a "con 20 variables el GP ve poco".

{'k': 3, 'factor': 0.5, 'perdida_val': 0.26521, 'epocas': 29, 'segundos': 71}
{'k': 3, 'factor': 1, 'perdida_val': 0.26529, 'epocas': 11, 'segundos': 21}
{'k': 3, 'factor': 2, 'perdida_val': 0.26592, 'epocas': 18, 'segundos': 35}
{'k': 5, 'factor': 0.5, 'perdida_val': 0.25492, 'epocas': 19, 'segundos': 37}
{'k': 5, 'factor': 1, 'perdida_val': 0.25415, 'epocas': 17, 'segundos': 34}
{'k': 5, 'factor': 2, 'perdida_val': 0.25484, 'epocas': 14, 'segundos': 28}
{'k': 10, 'factor': 0.5, 'perdida_val': 0.25286, 'epocas': 17, 'segundos': 34}
{'k': 10, 'factor': 1, 'perdida_val': 0.25141, 'epocas': 19, 'segundos': 36}
{'k': 10, 'factor': 2, 'perdida_val': 0.25231, 'epocas': 14, 'segundos': 27}
{'k': 20, 'factor': 0.5, 'perdida_val': 0.25285, 'epocas': 20, 'segundos': 39}
{'k': 20, 'factor': 1, 'perdida_val': 0.24949, 'epocas': 20, 'segundos': 40}
{'k': 20, 'factor': 2, 'perdida_val': 0.24973, 'epocas': 29, 'segundos': 56}
     k  factor  perdida_val  epocas  segundos
10  20     1.0      0.24949 

In [21]:
# Bloque 5. El modelo final: el GP con la mejor (k, scale), entrenado con TODO train y early stopping en validación.
# Después, una pasada por train arma la matriz de precisión P (y su Cholesky) y predecimos logit y varianza para
# validación y pool. Teoría en dos líneas: el GP es una regresión logística sobre D = 1024 random features
# φ(x) = √(2/D) cos(Wx + b), que aproximan un kernel gaussiano; la aproximación de Laplace da la precisión
# P = I + Σ p̂(1−p̂) φφᵀ, y la incertidumbre de cada solicitante es la varianza φ(x)ᵀ P⁻¹ φ(x) (ecuación 4 del informe).

from sklearn.metrics import log_loss, roc_auc_score
from redes import ajustar_covarianza, predecir

mejor = tabla_busqueda.iloc[0]   # la primera fila = la de menor pérdida de validación
k, factor = int(mejor["k"]), float(mejor["factor"])
cols = orden[:k]
print("elegido: k =", k, "| scale =", factor, "x raíz(k) | variables:", nombres[cols].tolist())

tr, va = armar_loaders(cols)   # ahora con TODO train
torch.manual_seed(SEED)
gp = GPVariables(k, scale=factor * math.sqrt(k))
hist = entrenar(gp, tr, va, lr=LR)   # con verbose (por defecto): imprime la pérdida de cada época
# loader de todo train, sin mezclar y en tandas de 4096, para la pasada de la covarianza
loader_cov = DataLoader(TensorDataset(torch.from_numpy(X_tr[:, cols]), torch.from_numpy(y_tr)), batch_size=4096)
ajustar_covarianza(gp, loader_cov)   # arma la matriz P (la "memoria" de por dónde pasaron los datos) y su Cholesky
logit_va, var_va = predecir(gp, torch.from_numpy(X_va[:, cols]))   # logit de mora y varianza en validación
logit_po, var_po = predecir(gp, torch.from_numpy(X_po[:, cols]))   # y en el pool

# La pérdida de validación baja de 0.2522 (época 1) a 0.2487 (época 9, la mejor) y el early stopping corta en la 14.
# Train 0.2495 contra validación 0.2487: no hay sobreajuste. Queda cargado el modelo de la época 9, no el último.

elegido: k = 20 | scale = 1.0 x raíz(k) | variables: ['num__AMT_GOODS_PRICE', 'num__AMT_CREDIT', 'num__EXT_SOURCE_3', 'num__EXT_SOURCE_2', 'cat__CODE_GENDER_F', 'cat__NAME_EDUCATION_TYPE_Higher education', 'num__EXT_SOURCE_1', 'cat__NAME_CONTRACT_TYPE_Revolving loans', 'cat__NAME_EDUCATION_TYPE_Incomplete higher', 'cat__ORGANIZATION_TYPE_Self-employed', 'num__DAYS_EMPLOYED', 'cat__ORGANIZATION_TYPE_Business Entity Type 3', 'cat__NAME_FAMILY_STATUS_Married', 'cat__ORGANIZATION_TYPE_Construction', 'num__missingindicator_OWN_CAR_AGE', 'cat__OCCUPATION_TYPE_Core staff', 'num__FLAG_DOCUMENT_3', 'num__AMT_ANNUITY', 'num__missingindicator_EXT_SOURCE_1', 'num__DEF_30_CNT_SOCIAL_CIRCLE']
época  1 | pérdida train 0.2624 | pérdida val 0.2522
época  2 | pérdida train 0.2519 | pérdida val 0.2499
época  3 | pérdida train 0.2512 | pérdida val 0.2497
época  4 | pérdida train 0.2508 | pérdida val 0.2503
época  5 | pérdida train 0.2506 | pérdida val 0.2492
época  6 | pérdida train 0.2503 | pérdida val 0

In [22]:
# Bloque 6. Cierre del 03: medimos la calidad del GP en validación (AUC y log-loss por segmento, comparando con
# XGBoost), chequeamos que las varianzas tengan sentido (positivas, y la mediana thin-file contra el resto) y
# guardamos las predicciones con el mismo formato que las de XGBoost (SK_ID_CURR, thin_file, TARGET, logit, var).
# Estos CSV son lo único que necesita el 04: entrenamos una vez, guardamos predicciones y experimentamos sobre ellas.

def prob(z):
    """Sigmoide: pasa de logit a probabilidad de mora."""
    return 1 / (1 + np.exp(-z))
print("Validación (AUC | log-loss):")
for nombre, z in [("XGBoost", xgb_va["logit"].to_numpy()), ("GP", logit_va)]:
    # np.ones_like(thin_va) = todos True (el total); thin_va = solo thin-file; ~thin_va = el resto
    for seg, m in [("total", np.ones_like(thin_va)), ("thin-file", thin_va), ("resto", ~thin_va)]:
        auc = roc_auc_score(y_va[m], z[m])   # AUC: qué tan bien ordena (solo importa el orden)
        ll = log_loss(y_va[m], prob(z[m]))   # log-loss: además castiga probabilidades mal calibradas
        print(f"  {nombre:8s} {seg:10s} AUC {auc:.3f} | log-loss {ll:.4f}")
print("\nVarianzas en el pool:")
print("  todas positivas:", bool((var_po > 0).all()),
      "| mínimo", var_po.min(), "| máximo", var_po.max())
print("  mediana thin-file:", np.median(var_po[thin_po]),
      "| mediana resto:", np.median(var_po[~thin_po]))
# guardamos una tabla por partición con el mismo formato que las de XGBoost
for nombre, ids, thin, y, z, v in [("calib", ids_va, thin_va, y_va, logit_va, var_va),
                                   ("test", ids_po, thin_po, y_po, logit_po, var_po)]:
    tabla = pd.DataFrame({"SK_ID_CURR": ids, "thin_file": thin.astype(int),
                          "TARGET": y.astype(int), "logit": z, "var": v})
    tabla.to_csv(f"outputs/preds/gp_vars_{nombre}.csv", index=False)   # index=False: sin la columna del índice
print("\nGuardado: outputs/preds/gp_vars_calib.csv y gp_vars_test.csv")

# Validación (AUC | log-loss): XGBoost total 0.762 | thin 0.725 | resto 0.767; GP total 0.751 | thin 0.714 | resto
# 0.758. El GP (20 variables) queda ~0.01 de AUC abajo de XGBoost (200 variables), lo esperable: los árboles suelen
# ganarle a las redes y a los GP en datos tabulares, y además el GP ve 10 veces menos variables.
# Varianzas: todas positivas. Y acá viene lo interesante: la mediana de los thin-file (0.0114) es MENOR que la del
# resto (0.0144), un cociente de 0.79. Es lo contrario de la hipótesis: con estas 20 variables, el GP no ve a los
# thin-file como más lejanos. Ese 0,79 es uno de los cinco cocientes de la sección 4.1 del informe.
# Una lectura posible (no verificada): las 20 variables casi no distinguen a los thin-file, y además en P cada
# solicitante pesa p(1−p), así que la varianza también depende del nivel de riesgo y no solo de la distancia.
# Por eso la apuesta pasa a ser la SNGP (05), que ve las 200 variables.

Validación (AUC | log-loss):
  XGBoost  total      AUC 0.762 | log-loss 0.2452
  XGBoost  thin-file  AUC 0.725 | log-loss 0.2964
  XGBoost  resto      AUC 0.767 | log-loss 0.2366
  GP       total      AUC 0.751 | log-loss 0.2487
  GP       thin-file  AUC 0.714 | log-loss 0.3028
  GP       resto      AUC 0.758 | log-loss 0.2396

Varianzas en el pool:
  todas positivas: True | mínimo 0.0014770157 | máximo 0.34290704
  mediana thin-file: 0.011449933 | mediana resto: 0.0144176725

Guardado: outputs/preds/gp_vars_calib.csv y gp_vars_test.csv
